# Base vs. RL pass@k at 1.5B (Colab)

Qwen2.5-1.5B vs. Qwen-2.5-1.5B-SimpleRL-Zoo on the 200-problem stratified MATH500 subset,
n = 128 samples per problem, temperature 0.6, top-p 0.95, up to 16k tokens (same setup as the
7B main experiment on HPC).

**Runtime:** Runtime → Change runtime type → **L4** or **A100** GPU (T4 works but falls back to float16).

**Run cells 1–6 in order.** Cell 3 builds a separate Python 3.10 environment (Colab's own Python is too
new for vLLM 0.6.3); it takes ~5 min the first time and is skipped afterwards on the same VM. No runtime
restart is needed.

**After a disconnect:** reconnect and run all cells again. Finished chunks are saved to Google Drive and
skipped; cell 3 skips the install unless the VM was reset.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 2. Google Drive (results live here so they survive disconnects) + code
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = 'https://github.com/YujunLuviaYao/limit-of-RLVR.git'   # private repo: see note below
BRANCH = 'claude/hopeful-fermat-5zkwvb'
REPO = '/content/limit-of-RLVR'
OUT_ROOT = '/content/drive/MyDrive/rlvr/main_out_1.5B'
os.makedirs(OUT_ROOT, exist_ok=True)

if not os.path.isdir(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} pull --ff-only
# If the repo is private, clone with a token instead, e.g.
#   from google.colab import userdata
#   REPO_URL = f"https://{userdata.get('GH_TOKEN')}@github.com/YujunLuviaYao/limit-of-RLVR.git"
# (add GH_TOKEN under the key icon "Secrets" in the left sidebar; never paste it into a cell).

In [ ]:
# 3. Python 3.10 environment with vLLM 0.6.3 (once per VM, ~5 min; skipped if already done).
# Colab's Python is too new for vLLM 0.6.3 (it needs torch 2.4.0, which has no wheels for it),
# so we build a separate venv with uv. The notebook kernel itself is not touched.
import os, subprocess
VENV = '/content/py310'
if not os.path.exists(f'{VENV}/.ready'):
    !pip install -q uv
    !rm -rf {VENV} && uv venv {VENV} --python 3.10
    !uv pip install --python {VENV}/bin/python vllm==0.6.3 "transformers<4.48" "math-verify[antlr4_11_0]==0.6.0" word2number Pebble sympy==1.12 timeout-decorator jieba datasets matplotlib huggingface_hub
    # latex2sympy (authors' grader) upgrades antlr4 from 4.11.0 to 4.11.1, as in the authors' setup.
    !uv pip install --python {VENV}/bin/python -e /content/limit-of-RLVR/math/examples/math_eval/latex2sympy

check = subprocess.run(
    [f'{VENV}/bin/python', '-c',
     "import vllm, torch, transformers, huggingface_hub, importlib.metadata as m; "
     "print('vllm', vllm.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, "
     "'| huggingface_hub', huggingface_hub.__version__, '| antlr4', m.version('antlr4-python3-runtime'))"],
    capture_output=True, text=True)
if check.returncode != 0:
    print(check.stderr[-3000:])
    raise RuntimeError('Environment check failed: see the error above (and the install output).')
print(check.stdout)
open(f'{VENV}/.ready', 'w').close()

In [ ]:
# 4. Settings (re-run after every restart/reconnect)
import os, subprocess
REPO = '/content/limit-of-RLVR'
MATH = f'{REPO}/math'
OUT_ROOT = '/content/drive/MyDrive/rlvr/main_out_1.5B'
MODELS = {
    'Qwen2.5-1.5B': 'Qwen/Qwen2.5-1.5B',
    'SimpleRL-Zoo-1.5B': 'hkust-nlp/Qwen-2.5-1.5B-SimpleRL-Zoo',
}

# Scripts call `python`: put the Python 3.10 venv first on PATH.
VENV = '/content/py310'
if f'{VENV}/bin' not in os.environ['PATH'].split(':'):
    os.environ['PATH'] = f"{VENV}/bin:" + os.environ['PATH']

os.environ['MODELS_DIR'] = '/content/models'       # local disk: fast, re-downloaded after a VM reset
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['OUT_ROOT'] = OUT_ROOT
os.environ['CHUNK_SIZE'] = '25'                    # smaller chunks = less lost on a disconnect
os.environ['N_SAMPLING'] = '128'

cap = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
if cap and float(cap) < 8.0:
    os.environ['VLLM_DTYPE'] = 'half'
    print(f'GPU compute capability {cap}: no bfloat16, using float16 (note this in the report).')
else:
    os.environ.pop('VLLM_DTYPE', None)
    print(f'GPU compute capability {cap}: using the model dtype (bfloat16).')

In [ ]:
# 5. Run both models (resumable: re-run this cell after a disconnect)
for name, repo in MODELS.items():
    !cd {MATH} && bash run_main.sh {name} {repo}

In [ ]:
# 6. pass@k + plot
from IPython.display import Image, display
args = ' '.join(f"'{name}={OUT_ROOT}/{name}/chunk_*/math500/*.jsonl'" for name in MODELS)
!cd {MATH} && python passk_minimal.py --plot {OUT_ROOT}/passk_1.5B.png {args}
display(Image(f'{OUT_ROOT}/passk_1.5B.png'))